In [18]:
!pip install google-genai openai requests python-dotenv
# Note: Si le package 'agents' provient d'un dépôt spécifique ou d'une bibliothèque interne
# comme 'openai-agents' (utilisé dans le SDK d'OpenAI), nous installons également les prérequis.
# Installons l'agent SDK d'OpenAI s'il s'agit de la bibliothèque mentionnée dans vos commentaires :
!pip install openai-agents

In [19]:
# 1. Installer zstd (requis par l'installateur Ollama) puis installer Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Démarrer le serveur Ollama en arrière-plan
import subprocess
import time

print("Démarrage d'Ollama...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Laisser le temps au serveur de s'initialiser

# 3. Télécharger le modèle gemma2 de taille réduite pour Colab
# !ollama pull gemma2:2b

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Hit:9 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,177 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Fetched 2,330 kB in 2s (1,087 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelea

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it.
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying.
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

In [20]:
# -------------------------------------------------------------------
# ADVANCED SOLUTION FOR THE EXERCISE (Cloud Ollama Setup)
# -------------------------------------------------------------------
from dotenv import load_dotenv
import os
import re
import html
import asyncio
import requests
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    SQLiteSession,
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from google.colab import userdata

load_dotenv(override=True)

# -------------------------------------------------------------------
# 1) Set up the cloud model connection
# -------------------------------------------------------------------
# Key created at https://ollama.com/settings/keys, stored in Colab Secrets
# (left sidebar, key icon) as OLLAMA_API_KEY, with "Notebook access" switched ON.
OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")

# Ollama's hosted API (OpenAI-compatible). No local server needed.
OLLAMA_BASE_URL = "https://ollama.com/v1"

# On the hosted API the model name has NO "-cloud" suffix.
MODEL_NAME = "gemma4:31b"

ollama_client = AsyncOpenAI(api_key=OLLAMA_API_KEY, base_url=OLLAMA_BASE_URL)
set_default_openai_client(ollama_client)
set_default_openai_api("chat_completions")
set_tracing_disabled(True)

In [21]:
# -------------------------------------------------------------------
# 2) Configure Telegram (instead of Resend)
# -------------------------------------------------------------------
# Telegram bots need no domain and no DNS. Setup (one time):
#   1. In Telegram, talk to @BotFather -> /newbot -> copy the token
#   2. Put it in .env:  TELEGRAM_BOT_TOKEN=123456:ABC...
#   3. Open your new bot in Telegram and press Start / send "hello"
#   4. Run get_chat_id() below, then put it in .env:  TELEGRAM_CHAT_ID=123456789
TELEGRAM_BOT_TOKEN = userdata.get('TELEGRAM_BOT_TOKEN')
# TELEGRAM_CHAT_ID = userdata.get("TELEGRAM_CHAT_ID")
TELEGRAM_API = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}"
print(bool(userdata.get('TELEGRAM_BOT_TOKEN')))
print(requests.get(f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/getMe", timeout=15).json())


# Safety: avoid infinite loops when the SDR keeps replying.
MAX_SDR_REPLIES = 5


def get_chat_id() -> str | None:
    """Return the chat id of the last person who messaged the bot."""
    r = requests.get(f"{TELEGRAM_API}/getUpdates", timeout=30).json()
    if not r.get("ok") or not r["result"]:
        print("No messages yet - open your bot in Telegram and send it 'hello' first.")
        return None
    chat_id = str(r["result"][-1]["message"]["chat"]["id"])
    print("Your chat id:", chat_id)
    return chat_id

True
{'ok': True, 'result': {'id': 9051989885, 'is_bot': True, 'first_name': 'send_and_recive_bot', 'username': 'agent_sendrecive_Bot', 'can_join_groups': True, 'can_read_all_group_messages': False, 'supports_inline_queries': False, 'supports_guest_queries': False, 'can_connect_to_business': False, 'has_main_web_app': False, 'has_topics_enabled': False, 'allows_users_to_create_topics': False, 'can_manage_bots': False, 'supports_join_request_queries': False}}


In [22]:
r = requests.get(f"{TELEGRAM_API}/getUpdates", timeout=30).json()
for u in r["result"]:
    m = u["message"]
    print(m["chat"]["id"], m["from"]["first_name"], m["from"]["is_bot"], m["text"])


In [23]:
# -------------------------------------------------------------------
# 3) Define the agents
# -------------------------------------------------------------------
# This is the same idea as the earlier notebook:
# - 3 sales agents with different writing styles
# - 1 picker to choose the best email
# - 1 critic to review it
# - 1 refiner to fix it
# - 1 SDR reply engine for incoming responses
INTRO = """
You are a sales agent working for ComplAI,
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.
You write emails. Always start with a line 'Subject: ...' followed by a blank line and the body.
Sign the email as 'Alex, ComplAI'. Never leave placeholders such as [Name] - use the prospect info you are given.
"""

sales_agent1 = Agent(
    name="Professional Sales Agent",
    model=MODEL_NAME,
    instructions=INTRO + "Your email style is professional, serious, with gravitas and credibility.",
)

sales_agent2 = Agent(
    name="Humorous Sales Agent",
    model=MODEL_NAME,
    instructions=INTRO + "Your email style is witty, engaging, and humorous.",
)

sales_agent3 = Agent(
    name="Executive Sales Agent",
    model=MODEL_NAME,
    instructions=INTRO + "Your email style is concise, to the point, in the style of a busy senior executive.",
)

sales_picker = Agent(
    name="Sales Picker",
    model=MODEL_NAME,
    instructions="""
You pick the best cold sales email from the given options.
Imagine you are the customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only (including its Subject line).
""",
)

# This critic is strict on quality. We ask it to respond with exactly "APPROVED"
# or with short improvement notes. That makes orchestration by code easy.
email_critic = Agent(
    name="Email Critic",
    model=MODEL_NAME,
    instructions="""
You are a demanding sales-email reviewer.
Check the email for: a compelling subject line, a clear value proposition, a single clear call to action,
a reasonable length (under 150 words) and no placeholders like [Name].
- If the email is good enough to send, reply with exactly one word: APPROVED
- Otherwise reply with 2-4 short, concrete bullet points of what to improve. Do not rewrite the email.
""",
)

email_refiner = Agent(
    name="Email Refiner",
    model=MODEL_NAME,
    instructions="""
You improve cold sales emails.
You receive an email and feedback from a reviewer. Rewrite the email applying the feedback.
Reply with the improved email only (starting with 'Subject: ...'), no commentary.
""",
)

# This agent handles the reply from a prospect after they answer the first email.
sdr_replier = Agent(
    name="SDR",
    model=MODEL_NAME,
    instructions="""
You are Alex, a Sales Development Rep at ComplAI (AI-powered SOC2 compliance SaaS).
A prospect replied to your cold email. Write the reply email.
Rules:
- Answer their questions honestly; do not invent pricing or features you are not sure about.
- If they are interested, propose a 15-minute call and suggest two concrete time slots.
- If they say they are not interested / unsubscribe, thank them politely and do NOT push.
- Keep it short (under 120 words), friendly, human. Sign as 'Alex, ComplAI'.
Reply with the BODY of the email only (no 'Subject:' line).
""",
)




In [27]:
# -------------------------------------------------------------------
# 4) Helper functions
# -------------------------------------------------------------------
def split_subject(email_text: str) -> tuple[str, str]:
    """
    The agents answer in the form:
        Subject: ...

        body...
    We split it into subject and body for sending.
    """
    match = re.match(r"\s*\**Subject:\**\s*(.+?)\s*\n+(.*)", email_text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip(" *"), match.group(2).strip()
    return "Quick question about SOC2", email_text.strip()


def text_to_html(text: str) -> str:
    """Simple plain-text to HTML conversion for email clients."""
    return "<html><body>" + html.escape(text).replace("\n", "<br>") + "</body></html>"


def strip_quoted_text(body: str) -> str:
    """
    Real email replies often contain previous thread text such as:
        On Mon, Bob wrote:
        > previous message
    We remove that quoted history so the SDR only responds to the new message.
    """
    lines = []
    for line in body.splitlines():
        if re.match(r"^\s*On .+ wrote:\s*$", line) or line.startswith(">") or line.startswith("From:"):
            break
        lines.append(line)
    return "\n".join(lines).strip()

In [28]:
# -------------------------------------------------------------------
# 5) Orchestration by Code
# -------------------------------------------------------------------
# This is where Python decides the workflow.
# We explicitly define the process:
#   3 writers in parallel -> picker -> critic -> optional refiner loop -> final email
async def draft_and_refine(prospect: str, max_rounds: int = 3, verbose: bool = True) -> str:
    """
    This is orchestration by code: the developer controls the sequence.

    Flow:
        3 writers (in parallel) -> picker -> critic -> refiner (if needed)
    """
    prompt = f"Write a cold sales email to this prospect: {prospect}"

    with trace("Draft and refine"):
        # STEP 1: run all 3 writers at the same time
        drafts = await asyncio.gather(
            Runner.run(sales_agent1, prompt),
            Runner.run(sales_agent2, prompt),
            Runner.run(sales_agent3, prompt),
        )
        drafts_text = [d.final_output for d in drafts]

        # STEP 2: bundle the drafts and let the picker pick the best one
        bundle = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(drafts_text)
        print(bundle)
        best = (await Runner.run(sales_picker, bundle)).final_output
        if verbose:
            print("=== PICKED DRAFT ===\n" + best)

        # STEP 3: review/refine loop
        # The Python code decides when to stop, not the LLM
        for round_number in range(1, max_rounds + 1):
            review = (await Runner.run(email_critic, best)).final_output.strip()
            if verbose:
                print(f"\n--- Round {round_number}: critic says ---\n{review}")

            # Stop once the critic approves the draft
            if review.upper().startswith("APPROVED"):
                break

            # Rewrite using the reviewer feedback
            refine_input = f"EMAIL:\n{best}\n\nFEEDBACK:\n{review}"
            best = (await Runner.run(email_refiner, refine_input)).final_output
            if verbose:
                print(f"\n=== REFINED DRAFT (round {round_number}) ===\n{best}")

    return best




In [29]:
# -------------------------------------------------------------------
# 6) Sending via Telegram
# -------------------------------------------------------------------
def send_via_telegram(chat_id: str, subject: str, text_body: str) -> str:
    """
    Send a message with the Telegram Bot API.
    Returns a status string (and prints errors so they are never silent).
    """
    if not TELEGRAM_BOT_TOKEN:
        print(f"[DRY-RUN - no TELEGRAM_BOT_TOKEN]\nTo: {chat_id}\nSubject: {subject}\n\n{text_body}\n")
        return "dry-run: message printed, not sent"

    text = f"{subject}\n\n{text_body}" if subject else text_body
    response = requests.post(
        f"{TELEGRAM_API}/sendMessage",
        json={"chat_id": chat_id, "text": text[:4096]},  # Telegram limit: 4096 chars
        timeout=30,
    )
    if response.status_code >= 300:
        msg = f"ERROR {response.status_code}: {response.text}"
        print(msg)
        return msg
    return f"sent (message_id={response.json()['result']['message_id']})"

In [35]:
# -------------------------------------------------------------------
# 7) Orchestration by LLM
# -------------------------------------------------------------------
# Here the manager agent decides which tool to call and when.
def build_sales_manager(chat_id: str) -> Agent:
    """
    This agent:
    1. Calls the 3 writer agents
    2. Chooses the best draft
    3. Sends the draft to the critic
    4. Refines if necessary
    5. Sends only one final message
    """

    @function_tool
    def send_email_tool(subject: str, body: str) -> str:
        """Send the final sales email to the prospect (via Telegram)."""
        return send_via_telegram(chat_id, subject, body)

    description = "Write a cold sales email for the prospect described in the input."

    tools = [
        sales_agent1.as_tool("sales_email_writer_1", description),
        sales_agent2.as_tool("sales_email_writer_2", description),
        sales_agent3.as_tool("sales_email_writer_3", description),
        email_critic.as_tool("email_critic", "Review an email. Returns APPROVED or improvement feedback."),
        email_refiner.as_tool("email_refiner", "Rewrite an email. Input must contain the EMAIL and the FEEDBACK."),
        send_email_tool,
    ]

    return Agent(
        name="Sales Manager",
        model=MODEL_NAME,
        tools=tools,
        instructions="""
You are a Sales Manager at ComplAI. Follow these steps in order:
1. Ask each of the three sales_email_writer tools for a draft, giving them the prospect description.
2. Pick the best draft.
3. Send it to email_critic. If the answer is not 'APPROVED', give the email AND the feedback to email_refiner,
   then review the result again. Do at most 2 refinement rounds.
4. Use send_email_tool ONCE with the final email (subject and body). Never send more than one email.
5. If send_email_tool returns an error, report the exact error. Do not claim success.
""",
    )

In [36]:
# -------------------------------------------------------------------
# 8) Handling a prospect reply (Telegram polling)
# -------------------------------------------------------------------
async def handle_incoming_reply(chat_id: str, body: str) -> str:
    """
    - Load the conversation memory for this chat
    - Use the SDR agent to respond
    - Send the reply back on Telegram
    """
    session = SQLiteSession(session_id=f"tg-{chat_id}", db_path="sdr_conversations.db")

    history = await session.get_items()
    sdr_replies = sum(1 for item in history if item.get("role") == "assistant")
    if sdr_replies >= MAX_SDR_REPLIES:
        return f"ignored: reached the limit of {MAX_SDR_REPLIES} automatic replies"

    result = await Runner.run(sdr_replier, f"The prospect wrote:\n\n{body}", session=session)
    reply_body = result.final_output.strip()

    status = send_via_telegram(chat_id, "", reply_body)
    return f"replied -> {status}"


async def seed_conversation(chat_id: str, first_email: str):
    """Store the first outbound email so the SDR has context when the prospect replies."""
    session = SQLiteSession(session_id=f"tg-{chat_id}", db_path="sdr_conversations.db")
    await session.add_items([{"role": "assistant", "content": first_email}])


async def listen_for_replies(chat_id: str, duration_seconds: int = 120):
    """
    Long-poll Telegram for new messages and let the SDR answer them.
    Works from a notebook: no public URL or webhook needed.
    Stop it early with the Interrupt button.
    """
    # getUpdates does not work while a webhook is set, so clear any
    await asyncio.to_thread(requests.post, f"{TELEGRAM_API}/deleteWebhook", timeout=30)

    # Skip old messages: start after the latest existing update
    old = (await asyncio.to_thread(requests.get, f"{TELEGRAM_API}/getUpdates", timeout=30)).json()
    offset = (old["result"][-1]["update_id"] + 1) if old.get("result") else None

    print(f"Listening for replies for {duration_seconds}s - answer the bot in Telegram...")
    loop = asyncio.get_running_loop()
    end = loop.time() + duration_seconds
    while loop.time() < end:
        params = {"timeout": 20}
        if offset is not None:
            params["offset"] = offset
        resp = await asyncio.to_thread(
            requests.get, f"{TELEGRAM_API}/getUpdates", params=params, timeout=40
        )
        for update in resp.json().get("result", []):
            offset = update["update_id"] + 1
            msg = update.get("message") or {}
            text = msg.get("text")
            if not text or str(msg["chat"]["id"]) != str(chat_id):
                continue  # ignore other chats and non-text messages
            print(f"Prospect: {text}")
            print(await handle_incoming_reply(chat_id, text))
    print("Stopped listening.")

In [37]:
async def full_demo(listen_seconds: int = 120):
    # Récupérer dynamiquement le chat_id ou utiliser un secret si configuré
    try:
        chat_id = userdata.get('TELEGRAM_CHAT_ID')
    except Exception:
        chat_id = None

    if not chat_id:
        chat_id = get_chat_id()

    if not chat_id:
        print("Impossible de continuer sans Chat ID. Veuillez envoyer un message au Bot sur Telegram d'abord.")
        return

    prospect = "Dana, CTO at a 40-person fintech startup"

    # --- CODE-ORCHESTRATED VERSION ---
    final_email = await draft_and_refine(prospect, max_rounds=3, verbose=True)
    subject, body = split_subject(final_email)

    print("\n=== SENDING TO TELEGRAM ===")
    print(send_via_telegram(chat_id, subject, body))

    # Save the first outbound email so the SDR remembers it later
    await seed_conversation(chat_id, final_email)

    # Reply to the bot in Telegram and the SDR will answer
    await listen_for_replies(chat_id, duration_seconds=listen_seconds)

    # # --- LLM-ORCHESTRATED VERSION ---
    # manager = build_sales_manager(chat_id)
    # with trace("Sales manager"):
    #     result = await Runner.run(
    #         manager,
    #         "Write a cold sales email to Dana, CTO at a 40-person fintech startup."
    #     )
    #     print("\n=== MANAGER-ORCHESTRATED RESULT ===")
    #     print(result.final_output)

# Run the demo when this cell is executed
await full_demo()

Cold sales emails:

Subject: Scaling your security posture and SOC2 readiness

Dana,

As your team grows and your footprint in the fintech space expands, the complexity of maintaining a rigorous security posture increases proportionally. For a firm of 40 people, the manual overhead of gathering evidence and managing controls for SOC2 compliance often becomes a significant tax on engineering productivity.

ComplAI eliminates this friction. We provide an AI-powered SaaS platform specifically engineered to automate the continuous monitoring of your environment and streamline audit preparation. Rather than diverting your senior engineers toward manual documentation, our tool ensures you are audit-ready in real-time, reducing the risk of findings and accelerating your path to certification.

I would like to discuss how ComplAI can institutionalize your compliance efforts without slowing down your development velocity. Are you available for a brief conversation next Tuesday or Wednesday?

Al